## Statistical analysis of potential improvements at different training sizes for substituition datasets

In [1]:
import pandas as pd
from model_comparison import rm_tukey_hsd

### Read data

In [2]:
df_input = pd.read_csv("../simulation_outputs/substitutions/simulation_summary_all.csv")

# Make name shorter
df_input.loc[df_input["Model"] == "GaussianProcessRegressor", "Model"] = "GPR"
df_input.loc[df_input["Model"] == "ExtraTrees", "Model"] = "ET"

# Add simplified foundation column for grouping
df_input["Foundation Simple"] = df_input.apply(
    lambda row: ("" if not row['Baseline'] else " + Descriptors"),
    axis=1
)

df_input["Model Label"] = (
    df_input.apply(
        lambda row: f"{row['Model']}" + row['Foundation Simple'],
        axis=1
    )
)

df_input = df_input[
    df_input['Model Label'].isin([
        "GPR",
        "GPR + Descriptors"
    ])
]

df_input["method"] = df_input['Model Label']
df_input["cv_cycle"] = df_input["Seed"]
df_input["split"] = ["Random"]*len(df_input["method"])

datasets = ["A0A247D711_LISMN_Stadelmann_2021",
            "DN7A_SACS2_Tsuboyama_2023_1JIC",
            "ENVZ_ECOLI_Ghose_2023",
            "FKBP3_HUMAN_Tsuboyama_2023_2KFV",
            "MAFG_MOUSE_Tsuboyama_2023_1K1V",
            "POLG_PESV_Tsuboyama_2023_2MXD",
            "SBI_STAAM_Tsuboyama_2023_2JVG",
            "SDA_BACSU_Tsuboyama_2023_1PV0",
            "SOX30_HUMAN_Tsuboyama_2023_7JJK",
            "YNZC_BACSU_Tsuboyama_2023_2JVD"]

train_sizes = df_input["Train Size"].unique()

df_all = {data: {} for data in datasets}
for data in datasets:
    for size in train_sizes:
        df_all[data][size] = df_input[(df_input['Train Size'] == size) & (df_input["Dataset"] == data)]

# Defining column names for metrics
metric_ls = ["Test MSE","Test RMSE","Test MAE","Test R2"]

### Evaluation summary across datasets per training size

Takes the mean p-value and mean size of difference across datasets per training size

In [3]:
df_results = pd.DataFrame(columns=["Train Size", "Metric", "p-adj mean value", "p-adj std value", "effect size mean value", "effect size std value"])
df_results_all = pd.DataFrame(columns=["Train Size", "Dataset", "Metric", "p-adj", "effect size"])
for metric in metric_ls:
    for size in train_sizes:
        p_values = []
        effect_size = []
        for data in datasets:
            print(f"Dataset :{data} | Training size: {size}")
            df_all_temp = df_all[data][size]
            if metric == "Test R2":
                df_all_temp = df_all_temp[df_all_temp[metric] >= -10]
            print("df_all_temp: ", len(df_all_temp))

            results = rm_tukey_hsd(df_all_temp, metric, "Model Label")[0]
            p_values.append(results["p-adj"].iloc[0])
            effect_size.append(results["meandiff"].iloc[0])
            df_results_all.loc[len(df_results_all)] = [size, data, metric, results["p-adj"].iloc[0], results["meandiff"].iloc[0]]

        df_results.loc[len(df_results)] = [size, metric, pd.Series(p_values).mean(), pd.Series(p_values).std(), pd.Series(effect_size).mean(), pd.Series(effect_size).std()]

df_results = df_results.sort_values(by="Train Size").reset_index(drop=True)
df_results.to_csv("results/substitutions_results_summary.csv", index=False)
df_results_all.to_csv("results/substitutions_results.csv", index=False)

Dataset :A0A247D711_LISMN_Stadelmann_2021 | Training size: 50
df_all_temp:  20
Dataset :DN7A_SACS2_Tsuboyama_2023_1JIC | Training size: 50
df_all_temp:  20
Dataset :ENVZ_ECOLI_Ghose_2023 | Training size: 50
df_all_temp:  20
Dataset :FKBP3_HUMAN_Tsuboyama_2023_2KFV | Training size: 50
df_all_temp:  20
Dataset :MAFG_MOUSE_Tsuboyama_2023_1K1V | Training size: 50
df_all_temp:  20
Dataset :POLG_PESV_Tsuboyama_2023_2MXD | Training size: 50
df_all_temp:  20
Dataset :SBI_STAAM_Tsuboyama_2023_2JVG | Training size: 50
df_all_temp:  20
Dataset :SDA_BACSU_Tsuboyama_2023_1PV0 | Training size: 50
df_all_temp:  20
Dataset :SOX30_HUMAN_Tsuboyama_2023_7JJK | Training size: 50
df_all_temp:  20
Dataset :YNZC_BACSU_Tsuboyama_2023_2JVD | Training size: 50
df_all_temp:  20
Dataset :A0A247D711_LISMN_Stadelmann_2021 | Training size: 350
df_all_temp:  20
Dataset :DN7A_SACS2_Tsuboyama_2023_1JIC | Training size: 350
df_all_temp:  20
Dataset :ENVZ_ECOLI_Ghose_2023 | Training size: 350
df_all_temp:  20
Dataset :FK